# Week 1: Transactional Data Cleaning and Wrangling

## Objective

The objective of Week 1 is to prepare the raw transactional dataset for future cohort analysis.

Tasks:

- Load the transactional dataset
- Handle missing Customer IDs
- Remove refunded transactions
- Calculate Cohort Month for every customer

The cleaned dataset will be used in Week 2 for retention analysis.

In [38]:
import pandas as pd
df = pd.read_csv("OnlineRetail.csv", encoding='latin-1')


# Dataset Overview

Before cleaning the dataset, we inspect its structure and identify potential data quality issues.


In [39]:
print(df.head())
print(df.shape)

  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

        InvoiceDate  UnitPrice  CustomerID         Country  
0  12-01-2010 08:26       2.55     17850.0  United Kingdom  
1  12-01-2010 08:26       3.39     17850.0  United Kingdom  
2  12-01-2010 08:26       2.75     17850.0  United Kingdom  
3  12-01-2010 08:26       3.39     17850.0  United Kingdom  
4  12-01-2010 08:26       3.39     17850.0  United Kingdom  
(541909, 8)


In [ ]:

df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

mask = df['InvoiceNo'].astype(str).str.startswith('C')
df = df[~mask]
df = df.query("not InvoiceNo.str.startswith('C')", engine='python')
df = df.loc[~df['InvoiceNo'].astype(str).str.startswith('C')]
print("Dataset shape after removing cancelled transactions:", df.shape)
print("Sample InvoiceNo values:", df['InvoiceNo'].head().tolist())

Dataset shape after removing cancelled transactions: (532621, 8)
Sample InvoiceNo values: ['536365', '536365', '536365', '536365', '536365']


# Remove Missing Customer IDs

Customer retention and cohort analysis require unique customer identification.

Rows containing missing CustomerID values are removed because they cannot be associated with a specific customer.

In [40]:

df_cleaned = df.dropna(subset=['CustomerID'])
df_cleaned = df[df['CustomerID'].notna()]


# Display info about removed rows
print(f"Original dataset shape: {df.shape}")
print(f"Cleaned dataset shape: {df_cleaned.shape}")
print(f"Rows removed: {df.shape[0] - df_cleaned.shape[0]}")

Original dataset shape: (541909, 8)
Cleaned dataset shape: (406829, 8)
Rows removed: 135080


# Missing Value Analysis

Missing values can negatively affect data quality and analytical results.

This step identifies columns that contain null values and measures the extent of missing information.

In [7]:
#Display missing values
print("Missing values:\n", df_cleaned.isnull().sum())

df_cleaned = df_cleaned.dropna()
print("\nAfter droppinng nulls:", df_cleaned.shape)

Missing values:
 InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64

After droppinng nulls: (397924, 8)


In [8]:
# Display number of duplicated rows
print("Duplicates:", df_cleaned.duplicated().sum())

Duplicates: 5192


In [9]:
# Remove duplicated rows to avoid redundant cleaning operations and ensure accurate data assessment
df_cleaned = df_cleaned.drop_duplicates()
print("After removing duplicates:", df_cleaned.duplicated().sum())

After removing duplicates: 0


In [10]:
# Display Data type
print("Initial Data types:\n", 
      df_cleaned.dtypes)

Initial Data types:
 InvoiceNo          str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
UnitPrice      float64
CustomerID     float64
Country            str
dtype: object


# Identify Cancelled Transactions

Invoices beginning with the letter "C" represent cancelled orders.

These records do not indicate successful purchases and therefore should not be included in customer purchase analysis.

In [20]:
# Validate datatype by ensuring InvoiceDate is in datetime, Quantity is in number, and CustomerID is in string
# Convert InvoiceDate to Date Format
df_cleaned["InvoiceDate"] = pd.to_datetime(
    df_cleaned["InvoiceDate"])

# Convert Quantity to int64
df_cleaned["Quantity"] = pd.to_numeric(
    df_cleaned["Quantity"], errors="coerce")

# Convert CustomerID to string
df_cleaned["CustomerID"] = df_cleaned["CustomerID"].astype("int64").astype("str")

print("After Validation:\n", df_cleaned.dtypes)

After Validation:
 InvoiceNo                      str
StockCode                      str
Description                    str
Quantity                     int64
InvoiceDate         datetime64[us]
UnitPrice                  float64
CustomerID                     str
Country                        str
TransactionMonth         period[M]
CohortMonth              period[M]
dtype: object


# Remove Invalid Quantity Records

Transactions containing negative quantities are removed to retain only valid purchase records.

# Detect Invalid Unit Prices

Transactions with negative unit prices are identified because they do not represent valid sales.

# Remove Invalid Unit Prices

Records containing negative prices are removed to ensure consistency and reliability of revenue calculations.

In [21]:
# Display and Remove invlid quantity and price
print("Negative Quantity:", (df_cleaned["Quantity"]<0).sum())
print("Negative Unit Price:", (df_cleaned["UnitPrice"]<0).sum())

df_cleaned = df_cleaned[
    df_cleaned["Quantity"]>0]
df_cleaned = df_cleaned[
    df_cleaned["UnitPrice"]>0]
print("\nFinal shape:", df_cleaned.shape)

Negative Quantity: 0
Negative Unit Price: 0

Final shape: (392692, 10)


In [23]:
df_cleaned['InvoiceDate'] = pd.to_datetime(
    df_cleaned['InvoiceDate']
)

# Extract Transaction Month

The transaction month is extracted from the InvoiceDate column.

This feature is required for cohort analysis and customer retention tracking.

In [33]:
# Extract Transaction Month
df_cleaned['TransactionMonth'] = (
    df_cleaned['InvoiceDate']
    .dt.to_period('M')
)

print(df_cleaned[['InvoiceDate','TransactionMonth']].head())

          InvoiceDate TransactionMonth
0 2010-12-01 08:26:00          2010-12
1 2010-12-01 08:26:00          2010-12
2 2010-12-01 08:26:00          2010-12
3 2010-12-01 08:26:00          2010-12
4 2010-12-01 08:26:00          2010-12


# Calculate Customer Cohort Month

The first purchase month of each customer is identified.

This month becomes the customer's Cohort Month and serves as the basis for cohort analysis.

In [34]:
cohort_month = (
    df_cleaned.groupby('CustomerID')
    ['InvoiceDate']
    .min()
    .dt.to_period('M')
)

# Merge Cohort Information

The calculated Cohort Month is merged back into the dataset.

Each transaction now contains both the transaction month and the customer's cohort month.

In [35]:
df_cleaned['CohortMonth'] = (
    df_cleaned['CustomerID']
    .map(cohort_month)
)

# Preview Processed Dataset

The transformed dataset is displayed to verify that all cleaning and cohort preparation steps have been successfully applied.

In [36]:
df_cleaned[
    [
        'CustomerID',
        'InvoiceDate',
        'TransactionMonth',
        'CohortMonth'
    ]
].head(20)

,CustomerID,InvoiceDate,TransactionMonth,CohortMonth
0,17850,2010-12-01 08:26:00,2010-12,2010-12
1,17850,2010-12-01 08:26:00,2010-12,2010-12
2,17850,2010-12-01 08:26:00,2010-12,2010-12
3,17850,2010-12-01 08:26:00,2010-12,2010-12
4,17850,2010-12-01 08:26:00,2010-12,2010-12
5,17850,2010-12-01 08:26:00,2010-12,2010-12
6,17850,2010-12-01 08:26:00,2010-12,2010-12
7,17850,2010-12-01 08:28:00,2010-12,2010-12
8,17850,2010-12-01 08:28:00,2010-12,2010-12
9,13047,2010-12-01 08:34:00,2010-12,2010-12


# Export Cleaned Dataset

The final cleaned dataset is exported to a CSV file.

This file can now be used for exploratory data analysis, customer retention studies, cohort analysis, and business intelligence reporting.

In [37]:
df_cleaned.to_csv(
    "cleaned_retail_data.csv",
    index=False
)

print(
    "Cleaned dataset saved successfully."
)

Cleaned dataset saved successfully.


# Conclusion

The Online Retail dataset was successfully cleaned and prepared for further analysis.

During the preprocessing stage, missing customer records, cancelled transactions, duplicate entries, and invalid values were identified and removed. Data types were validated and standardized to ensure consistency across the dataset.

Additionally, transaction month and cohort month information were generated to support future customer retention and cohort analysis.

As a result, the final dataset is clean, reliable, and ready for exploratory data analysis, customer behavior analysis, cohort analysis, and business intelligence applications.